In [7]:
import os
import pandas as pd
import numpy as np
from glob import glob

# ====================================
# CONFIGURATION
# ====================================
BASE_DIR = "FARS"  # main folder with year subfolders
OUTPUT_PATH = "FARS/cleaned_data/fars_accident_cleaned.csv"

# Columns you previously used
VARS_OF_INTEREST = [
    "ST_CASE", "VE_TOTAL", "COUNTY", "COUNTYNAME", "CITY", "CITYNAME",
    "DAY", "MONTH", "MONTHNAME", "YEAR", "DAY_WEEK", "DAY_WEEKNAME",
    "HOUR", "RUR_URB", "RUR_URBNAME", "LATITUDE", "LONGITUD",
    "RELJCT2", "RELJCT2NAME", "TYP_INT", "TYP_INTNAME",
    "LGT_COND", "LGT_CONDNAME", "WEATHER", "WEATHERNAME"
]

# ====================================
# HELPERS (adapted from your original)
# ====================================
def replace_unknowns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Replace 'unknown' or 'not reported' (case-insensitive) with NaN."""
    for col in cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.lower()            # normalize to lowercase
                .replace(r"^(unknown|not reported)$", pd.NA, regex=True)
            )

    return df

def replace_invalid_coords(df: pd.DataFrame) -> pd.DataFrame:
    """Replace invalid longitude/latitude sentinel values with NaN."""
    invalid_values = {77.77770, 88.88880, 99.99990, 777.77770, 888.88880, 999.99990}
    if "LONGITUD" in df.columns:
        df["LONGITUD"] = pd.to_numeric(df["LONGITUD"], errors="coerce")
        df["LONGITUD"] = df["LONGITUD"].where(~df["LONGITUD"].isin(invalid_values), pd.NA)
    if "LATITUDE" in df.columns:
        df["LATITUDE"] = pd.to_numeric(df["LATITUDE"], errors="coerce")
        df["LATITUDE"] = df["LATITUDE"].where(~df["LATITUDE"].isin(invalid_values), pd.NA)
    return df

def clean_fars_accident_df(df: pd.DataFrame) -> pd.DataFrame:
    """Apply cleaning rules you provided to a single DataFrame (columns already uppercased)."""
    df = df.copy()

    # ID as you used previously
    # Ensure YEAR and ST_CASE exist (YEAR may be present from raw; we'll set it from folder earlier)
    if "YEAR" not in df.columns:
        raise KeyError("YEAR column missing from accident DataFrame before cleaning.")
    if "ST_CASE" not in df.columns:
        raise KeyError("ST_CASE column missing from accident DataFrame before cleaning.")

    df["ID"] = "FARS_" + df["YEAR"].astype(str) + "_" + df["ST_CASE"].astype(str)
    # also create CASE_ID to match other scripts
    #df["CASE_ID"] = df["YEAR"].astype(str) + "_" + df["ST_CASE"].astype(str)

    # Numeric/sentinel cleaning
    if "COUNTY" in df.columns:
        df["COUNTY"] = pd.to_numeric(df["COUNTY"], errors="coerce")
        df["COUNTY"] = df["COUNTY"].where(~df["COUNTY"].isin([998, 999]), pd.NA)
    if "CITY" in df.columns:
        df["CITY"] = pd.to_numeric(df["CITY"], errors="coerce")
        df["CITY"] = df["CITY"].where(~df["CITY"].isin([9898, 9999]), pd.NA)
    if "HOUR" in df.columns:
        df["HOUR"] = pd.to_numeric(df["HOUR"], errors="coerce")
        df["HOUR"] = df["HOUR"].replace(99, pd.NA)
    if "RUR_URB" in df.columns:
        df["RUR_URB"] = pd.to_numeric(df["RUR_URB"], errors="coerce")
        df["RUR_URB"] = df["RUR_URB"].where(~df["RUR_URB"].isin([8, 9]), pd.NA)
    if "RELJCT2" in df.columns:
        df["RELJCT2"] = pd.to_numeric(df["RELJCT2"], errors="coerce")
        df["RELJCT2"] = df["RELJCT2"].where(~df["RELJCT2"].isin([98, 99]), pd.NA)
    if "TYP_INT" in df.columns:
        df["TYP_INT"] = pd.to_numeric(df["TYP_INT"], errors="coerce")
        df["TYP_INT"] = df["TYP_INT"].where(~df["TYP_INT"].isin([98, 99]), pd.NA)
    if "LGT_COND" in df.columns:
        df["LGT_COND"] = pd.to_numeric(df["LGT_COND"], errors="coerce")
        df["LGT_COND"] = df["LGT_COND"].where(~df["LGT_COND"].isin([8, 9]), pd.NA)
    if "WEATHER" in df.columns:
        df["WEATHER"] = pd.to_numeric(df["WEATHER"], errors="coerce")
        df["WEATHER"] = df["WEATHER"].where(~df["WEATHER"].isin([98, 99]), pd.NA)

    # Handle textual "unknown"/"not reported"
    df = replace_unknowns(df, [
        "COUNTYNAME", "CITYNAME", "RUR_URBNAME",
        "RELJCT2NAME", "TYP_INTNAME", "LGT_CONDNAME", "WEATHERNAME"
    ])

    # Coordinates
    df = replace_invalid_coords(df)

    return df

# ====================================
# LOAD PER-YEAR FILES (folder structure)
# ====================================
def load_accident_file(year_folder: str) -> pd.DataFrame | None:
    """
    Load accident file inside a year folder. Accepts flexible capitalization.
    Returns DataFrame with uppercased column names and a YEAR column added.
    """
    # find file by pattern (case-insensitive)
    candidates = glob(os.path.join(year_folder, "[aA]ccident.csv"))
    if not candidates:
        # also accept ACCIDENT.CSV etc via case-insensitive glob on Windows may differ; try more patterns
        candidates = glob(os.path.join(year_folder, "accident.*"))
        candidates = [c for c in candidates if os.path.basename(c).lower().startswith("accident")]
    if not candidates:
        print(f"⚠️ No accident file found in {year_folder}")
        return None

    file_path = candidates[0]
    try:
        df = pd.read_csv(file_path, low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1", low_memory=False)

    # standardize column names to uppercase
    df.columns = df.columns.str.upper().str.strip()

    # subset to variables of interest if present (keep ST_CASE and YEAR regardless)
    cols = [c for c in VARS_OF_INTEREST if c in df.columns]
    # ensure ST_CASE and YEAR included if present in file; if not, we still require them later
    if "ST_CASE" in df.columns and "ST_CASE" not in cols:
        cols.append("ST_CASE")
    # Note: YEAR might be present in file, but we'll override it from the folder name below
    if "YEAR" in df.columns and "YEAR" not in cols:
        cols.append("YEAR")

    if cols:
        df = df[cols]

    # Add YEAR explicitly from folder name (overrides file YEAR)
    basename = os.path.basename(year_folder)
    # Try to parse year as int
    try:
        year_int = int(basename)
    except ValueError:
        # If folder name isn't numeric, try to find a 4-digit year substring
        import re
        m = re.search(r"(20\d{2})", basename)
        if m:
            year_int = int(m.group(1))
        else:
            raise ValueError(f"Cannot infer year from folder name: {year_folder}")
    df["YEAR"] = year_int

    return df

# ====================================
# COMBINE ALL YEARS
# ====================================
def combine_accident_data(base_dir: str) -> pd.DataFrame:
    """
    Walk year folders in base_dir, load accident files, clean each one,
    and produce one concatenated cleaned DataFrame.
    """
    year_folders = sorted([f.path for f in os.scandir(base_dir) if f.is_dir()])
    all_year_dfs = []

    for folder in year_folders:
        df = load_accident_file(folder)
        if df is None:
            continue

        # ensure ST_CASE exists for ID creation; if not, skip with a warning
        if "ST_CASE" not in df.columns:
            print(f"⚠️ ST_CASE missing in {folder} - skipping this year")
            continue

        # Clean
        df_clean = clean_fars_accident_df(df)

        # Keep only columns of interest + YEAR + CASE_ID + ID
        keep_cols = [c for c in VARS_OF_INTEREST if c in df_clean.columns]
        for extra in ["YEAR", "ID"]:
            if extra in df_clean.columns and extra not in keep_cols:
                keep_cols.append(extra)

        df_clean = df_clean[keep_cols]

        all_year_dfs.append(df_clean)
        print(f"✅ Processed {os.path.basename(folder)}: {df_clean.shape[0]} rows, {df_clean.shape[1]} cols")

    if not all_year_dfs:
        raise ValueError("No valid accident files found in base directory.")

    combined = pd.concat(all_year_dfs, ignore_index=True)

    # Final duplicate detection & dedupe by CASE_ID
    dup_mask = combined.duplicated(subset=["ID"], keep=False)
    n_dupes = int(dup_mask.sum())
    print(f"\n🔍 Duplicate check across combined dataset: found {n_dupes} duplicate rows by ID")
    if n_dupes > 0:
        # show first few duplicates
        print(combined.loc[dup_mask, ["ID"]].drop_duplicates().head())
        # keep first occurrence
        combined = combined.drop_duplicates(subset=["ID"], keep="first")

    # Reorder columns to place ID/YEAR first
    meta = [c for c in ["ID", "YEAR"] if c in combined.columns]
    other = [c for c in combined.columns if c not in meta]
    combined = combined[meta + other]

    return combined

# ====================================
# DATA QUALITY SUMMARY
# ====================================
def summarize_data_quality(df: pd.DataFrame):
    print("\n=== DATA QUALITY SUMMARY ===")
    missing_pct = df.isna().mean() * 100
    print("\nMissingness (%):")
    print(missing_pct.sort_values(ascending=False))
    print("\nRanges / Unique values (numeric columns):")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f" - {col}: min={df[col].min()}, max={df[col].max()}")
        else:
            print(f" - {col}: {df[col].nunique()} unique values")

# ====================================
# MAIN
# ====================================
if __name__ == "__main__":
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    combined_accident = combine_accident_data(BASE_DIR)
    summarize_data_quality(combined_accident)

    # final duplicate check on CASE_ID + optionally YEAR (should be unique)
    dupes_final = combined_accident.duplicated(subset=["ID"], keep=False)
    if dupes_final.any():
        print("\n⚠ Final duplicate rows remain (CASE_ID):")
        print(combined_accident.loc[dupes_final, ["ID"]].head())

    combined_accident.to_csv(OUTPUT_PATH, index=False)
    print(f"\n💾 Cleaned data saved to: {OUTPUT_PATH}")


✅ Processed 2016: 34748 rows, 26 cols
✅ Processed 2017: 34560 rows, 26 cols
✅ Processed 2018: 33919 rows, 26 cols
✅ Processed 2019: 33487 rows, 26 cols
✅ Processed 2020: 35935 rows, 26 cols
✅ Processed 2021: 39785 rows, 26 cols
✅ Processed 2022: 39422 rows, 26 cols
✅ Processed 2023: 37654 rows, 26 cols
⚠️ No accident file found in FARS\cleaned_data

🔍 Duplicate check across combined dataset: found 0 duplicate rows by ID

=== DATA QUALITY SUMMARY ===

Missingness (%):
WEATHER         5.747297
WEATHERNAME     5.378053
CITY            0.919485
CITYNAME        0.919485
HOUR            0.748161
LGT_COND        0.606542
LATITUDE        0.427964
LONGITUD        0.427964
TYP_INT         0.345066
LGT_CONDNAME    0.262167
TYP_INTNAME     0.251114
RELJCT2         0.246624
RUR_URBNAME     0.157853
RUR_URB         0.157853
RELJCT2NAME     0.149563
COUNTY          0.004490
COUNTYNAME      0.000691
YEAR            0.000000
DAY_WEEKNAME    0.000000
DAY_WEEK        0.000000
MONTHNAME       0.000000
MON